# Recover Gene Names for Unmapped Ensembl IDs

This notebook tries to map difficult/unmapped Ensembl gene IDs using multiple sources:

1. **Ensembl REST current release**
2. **Ensembl GRCh37 archive**
3. **MyGene.info**
4. **NCBI Entrez Gene** via MyGene Entrez IDs where available

It produces one combined Excel file showing the best recovered symbol/name and which source found it.


In [ ]:
!pip -q install pandas openpyxl requests mygene biopython


In [ ]:
import time
import requests
import pandas as pd
import mygene
from Bio import Entrez

# NCBI asks for an email when using Entrez.
# Replace this with your real email before heavy use.
Entrez.email = "your_email@example.com"

gene_ids = [
    "ENSG00000174112",
    "ENSG00000238684",
    "ENSG00000130724",
    "ENSG00000254185",
    "ENSG00000255323",
    "ENSG00000262371",
    "ENSG00000233865",
    "ENSG00000205665",
    "ENSG00000241861",
    "ENSG00000260662",
    "ENSG00000272374",
    "ENSG00000243492",
]

gene_ids = [x.strip().split(".")[0] for x in gene_ids]
gene_ids


## 1. Ensembl REST lookup

This checks Ensembl's current release and the GRCh37/hg19 archive.


In [ ]:
def query_ensembl_lookup(gene_id, server="https://rest.ensembl.org", source_label="ensembl_current"):
    url = f"{server}/lookup/id/{gene_id}"
    headers = {"Content-Type": "application/json"}
    params = {"expand": 0}
    try:
        r = requests.get(url, headers=headers, params=params, timeout=30)
        if r.status_code != 200:
            return {
                "ensembl_gene_id": gene_id,
                f"{source_label}_found": False,
                f"{source_label}_symbol": None,
                f"{source_label}_description": None,
                f"{source_label}_biotype": None,
                f"{source_label}_status": r.status_code,
            }
        data = r.json()
        return {
            "ensembl_gene_id": gene_id,
            f"{source_label}_found": True,
            f"{source_label}_symbol": data.get("display_name"),
            f"{source_label}_description": data.get("description"),
            f"{source_label}_biotype": data.get("biotype"),
            f"{source_label}_status": r.status_code,
        }
    except Exception as e:
        return {
            "ensembl_gene_id": gene_id,
            f"{source_label}_found": False,
            f"{source_label}_symbol": None,
            f"{source_label}_description": None,
            f"{source_label}_biotype": None,
            f"{source_label}_status": f"ERROR: {e}",
        }

current_df = pd.DataFrame([query_ensembl_lookup(gid) for gid in gene_ids])

grch37_df = pd.DataFrame([
    query_ensembl_lookup(
        gid,
        server="https://grch37.rest.ensembl.org",
        source_label="ensembl_grch37"
    )
    for gid in gene_ids
])

current_df


In [ ]:
grch37_df


## 2. MyGene.info backup

This aggregates several gene annotation resources and may recover IDs that Ensembl REST does not return cleanly.


In [ ]:
mg = mygene.MyGeneInfo()

mygene_results = mg.querymany(
    gene_ids,
    scopes="ensembl.gene",
    fields="symbol,name,type_of_gene,entrezgene,ensembl.gene,alias,summary",
    species="human",
    as_dataframe=False,
    returnall=False,
    verbose=False,
)

mygene_rows = []
for item in mygene_results:
    aliases = item.get("alias")
    if isinstance(aliases, list):
        aliases = "; ".join(map(str, aliases))
    mygene_rows.append({
        "ensembl_gene_id": item.get("query"),
        "mygene_found": not item.get("notfound", False),
        "mygene_symbol": item.get("symbol"),
        "mygene_name": item.get("name"),
        "mygene_type": item.get("type_of_gene"),
        "mygene_entrez_id": item.get("entrezgene"),
        "mygene_aliases": aliases,
        "mygene_summary": item.get("summary"),
        "mygene_id": item.get("_id"),
    })

mygene_df = pd.DataFrame(mygene_rows)
mygene_df


## 3. NCBI Gene backup

This only runs when MyGene.info returns an Entrez Gene ID.


In [ ]:
def query_ncbi_gene(entrez_id):
    if pd.isna(entrez_id) or entrez_id is None:
        return {"ncbi_found": False, "ncbi_symbol": None, "ncbi_description": None, "ncbi_status": "no_entrez_id"}
    try:
        handle = Entrez.esummary(db="gene", id=str(int(entrez_id)))
        record = Entrez.read(handle)
        handle.close()
        doc = record["DocumentSummarySet"]["DocumentSummary"][0]
        return {
            "ncbi_found": True,
            "ncbi_symbol": doc.get("Name"),
            "ncbi_description": doc.get("Description"),
            "ncbi_status": "ok",
        }
    except Exception as e:
        return {"ncbi_found": False, "ncbi_symbol": None, "ncbi_description": None, "ncbi_status": f"ERROR: {e}"}

ncbi_rows = []
for _, row in mygene_df.iterrows():
    out = query_ncbi_gene(row.get("mygene_entrez_id"))
    out["ensembl_gene_id"] = row["ensembl_gene_id"]
    ncbi_rows.append(out)
    time.sleep(0.34)

ncbi_df = pd.DataFrame(ncbi_rows)
ncbi_df


## 4. Combine results and choose best available mapping

Priority order:

1. Ensembl current symbol
2. Ensembl GRCh37 symbol
3. MyGene symbol
4. NCBI symbol


In [ ]:
combined = current_df.merge(grch37_df, on="ensembl_gene_id", how="outer")
combined = combined.merge(mygene_df, on="ensembl_gene_id", how="outer")
combined = combined.merge(ncbi_df, on="ensembl_gene_id", how="outer")

def first_nonempty(row, cols):
    for col in cols:
        val = row.get(col)
        if pd.notna(val) and str(val).strip() != "":
            return val
    return None

def source_for_first_nonempty(row, col_source_pairs):
    for col, source in col_source_pairs:
        val = row.get(col)
        if pd.notna(val) and str(val).strip() != "":
            return source
    return None

combined["best_symbol"] = combined.apply(
    lambda r: first_nonempty(r, [
        "ensembl_current_symbol",
        "ensembl_grch37_symbol",
        "mygene_symbol",
        "ncbi_symbol",
    ]), axis=1
)

combined["best_name_or_description"] = combined.apply(
    lambda r: first_nonempty(r, [
        "ensembl_current_description",
        "ensembl_grch37_description",
        "mygene_name",
        "ncbi_description",
    ]), axis=1
)

combined["best_biotype_or_type"] = combined.apply(
    lambda r: first_nonempty(r, [
        "ensembl_current_biotype",
        "ensembl_grch37_biotype",
        "mygene_type",
    ]), axis=1
)

combined["best_source"] = combined.apply(
    lambda r: source_for_first_nonempty(r, [
        ("ensembl_current_symbol", "Ensembl current"),
        ("ensembl_grch37_symbol", "Ensembl GRCh37"),
        ("mygene_symbol", "MyGene.info"),
        ("ncbi_symbol", "NCBI Gene via Entrez"),
    ]), axis=1
)

combined["mapped"] = combined["best_symbol"].notna()

summary_cols = [
    "ensembl_gene_id",
    "mapped",
    "best_symbol",
    "best_name_or_description",
    "best_biotype_or_type",
    "best_source",
    "ensembl_current_found",
    "ensembl_grch37_found",
    "mygene_found",
    "ncbi_found",
]

summary = combined[summary_cols].sort_values(["mapped", "ensembl_gene_id"], ascending=[False, True])
summary


## 5. Save results


In [ ]:
output_file = "unmapped_ensembl_gene_name_recovery.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="summary_best_mapping", index=False)
    combined.to_excel(writer, sheet_name="all_source_details", index=False)

summary.to_csv("unmapped_ensembl_gene_name_recovery_summary.csv", index=False)
combined.to_csv("unmapped_ensembl_gene_name_recovery_all_details.csv", index=False)

print("Saved:")
print(f"- {output_file}")
print("- unmapped_ensembl_gene_name_recovery_summary.csv")
print("- unmapped_ensembl_gene_name_recovery_all_details.csv")


## How to use the output

Use `best_symbol` for Venny only if `mapped = True`.

For IDs that remain unmapped, keep them in a separate sheet and do not mix them with gene symbols in Venny.
